# Module 2: Vector Search

## Embeddings

We use embedding models to convert text into vectors. In this example we use the Sentence-Transformers library to access the ["all-MiniLM-L6-v2" model](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2). 

In [1]:
from sentence_transformers import SentenceTransformer

In [2]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

We encode the sentence as a vector using the `encode` method

In [3]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [4]:
v1.shape

(384,)

In [5]:
v1

array([ 2.13903524e-02, -7.39799812e-02,  1.42065831e-03,  2.13816520e-02,
        2.45113187e-02,  3.15582566e-02, -1.10839725e-01, -1.05017498e-01,
       -6.18259050e-02, -6.42315438e-03,  3.72396712e-03,  9.06393453e-02,
       -9.49938595e-03,  6.53976724e-02,  1.10946922e-02, -2.10097395e-02,
       -3.35125700e-02, -4.31677103e-02,  9.96348076e-03,  1.41970133e-02,
       -6.40415549e-02, -7.04178913e-03, -7.91187957e-02,  5.80030642e-02,
        1.30210805e-03,  4.19732928e-03,  5.70979454e-02,  6.39447793e-02,
        2.49902755e-02, -3.95876393e-02, -3.79506014e-02,  2.70394590e-02,
        1.79423522e-02,  1.72272250e-02,  3.43311243e-02,  9.29055922e-03,
        5.86055182e-02, -4.97789830e-02, -5.05369110e-03,  4.34328318e-02,
       -1.56622883e-02, -2.97534503e-02, -5.13326144e-03,  5.13414815e-02,
        6.16062293e-03,  6.86980486e-02, -1.29505470e-02, -5.61938137e-02,
       -1.08265253e-02,  5.96683845e-02,  5.29940054e-02, -3.42754945e-02,
       -4.15274240e-02, -

In [6]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [7]:
v1.dot(dv)

np.float32(0.323324)

We can compare the two vectors by calculating the dot product of them. For the first query we get 0.32 showing some simliarity. 

For the second query, it has nothing to do with the document sentence so the results is closer to 0.

In [8]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [9]:
v2.dot(dv)

np.float32(0.019730479)

## Embedding our dataset

In [11]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py

--2026-06-23 08:34:39--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/ingest.py
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 888 [text/plain]
Saving to: ‘ingest.py’

ingest.py           100%[===================>]     888  --.-KB/s    in 0s      

2026-06-23 08:34:40 (160 MB/s) - ‘ingest.py’ saved [888/888]



In [12]:
from ingest import load_faq_data

In [13]:
documents = load_faq_data()

In [14]:
documents[0]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel.",
 'doc_id': '9e508f2212'}

We need to concatenate the question and answer field into single elements, before we complete the embeddings.

In [15]:
texts = []

for doc in documents:
    text = doc['question'] + " " + doc['answer']
    texts.append(text)

In [17]:
texts[0]

"Course: When does the course start? A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."

In [19]:
len(texts)

1350

Now we can encode each element as a vector. We will do this in batches.

In [18]:
from tqdm.auto import tqdm

In [20]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/27 [00:00<?, ?it/s]

1350

We can see the shape of our new vector store by converting it to a numpy arry, to which we can apply see the shape arg.

In [24]:
import numpy as np

In [25]:
X = np.array(vectors)

In [26]:
X.shape

(1350, 384)

## Vector Search

To perform the search, essentially we need to do the dot product of our search query with every vector in our list to find the vectors that produce the highest score. 

The manual way of doing this would be to create a scores list and do a forloop through the list of vectors, and append each score to our scores list. 

```python
scores = [v_query.dot(X[i]) for i in range(len(X))]
```

However, if we take our vector list and convert it into a numpy array it becomes a matrix which we can then use matrix multiplication to do the same operation much quicker and better optimised.

In [27]:
scores = X.dot(v1)

In [ ]:
idx = np.argmax(scores) # returns the index of the largest score

In [35]:
idx, scores[idx]

(np.int64(2), np.float32(0.76294106))

In [36]:
documents[2]

{'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute.",
 'doc_id': '3f1424af17'}

In [ ]:
top5 = np.argsort(scores)[-5:] # returns the index of the top 5 scores
top5 = top5[::-1] #sort from highest to lowest, as automatically it is the other way around

array([  2, 625, 907, 538,   7])

In [38]:
scores[top5[::-1]]

array([0.76294106, 0.75793713, 0.71921325, 0.6536312 , 0.56009984],
      dtype=float32)

In [42]:
# Have to loop through top5 indices to get documents

for idx in top5:
    print(documents[idx])

{'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.', 'doc_id': '068529125b'}
{'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.', 'doc_id': '74eb249bbf'}
{'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t 

To make the sorting process easier we can just add a `-` to the array argument of the argsort to sort in the opposite direction.

In [45]:
top5 = np.argsort(-scores)[:5]

In [46]:
top5

array([  2, 625, 907, 538,   7])